# 第10章 金融特征工程 —— 配套代码

对应正文 `book/part3/10-feature-engineering.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 演示内容

1. 环境初始化与数据加载
2. 价量特征：动量（多周期）、反转、波动率
3. 技术指标：MA/EMA、RSI、MACD、布林带（全部防前视）
4. 前视偏差对照实验：含前视 vs 无前视的 IC 对比
5. 截面 Winsorize 与 z-score 标准化
6. 特征矩阵构建与相关热力图
7. IC/ICIR 特征筛选与 IC 衰减曲线
8. 多重共线性 VIF 诊断
9. 习题参考解答（习题 10.1 ~ 10.5）

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1：环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()
rng = np.random.default_rng(42)

prices = load_sample_prices()
rets   = daily_returns(prices)
TICKERS = list(prices.columns)
TRADING_DAYS = 252

print(f'价格数据维度：{prices.shape}')
print(f'收益率数据维度：{rets.shape}')
print(f'交易日范围：{prices.index[0].date()} 至 {prices.index[-1].date()}')
print(f'股票列表：{TICKERS}')
prices.tail(3)

## 10.2 价量特征构造

| 特征 | 公式 | 含义 |
|---|---|---|
| 5/20/60 日动量 | $P_t/P_{t-n} - 1$ | 多周期价格趋势 |
| 5 日短期反转 | $-(P_t/P_{t-5} - 1)$ | 短期均值回归 |
| 20 日已实现波动率 | $\text{std}(r)\times\sqrt{252}$ | 波动率风险 |

> **注意**：所有特征均在构造后做 `.shift(1)`，确保预测 $t+1$ 期时只含 $\leq t$ 期信息。

In [ ]:
# Cell 2：价量特征（多周期动量、反转、波动率）

mom5  = prices.pct_change(5).shift(1)
mom20 = prices.pct_change(20).shift(1)
mom60 = prices.pct_change(60).shift(1)
rev5  = -prices.pct_change(5).shift(1)
log_rets = np.log(prices / prices.shift(1))
vol20 = log_rets.rolling(20).std().shift(1) * np.sqrt(TRADING_DAYS)

feat_preview = pd.concat(
    [mom5.add_suffix('_mom5'), mom20.add_suffix('_mom20'), vol20.add_suffix('_vol20')],
    axis=1
).dropna().round(4)
print(f'非 NaN 行数：{feat_preview.shape[0]}')
print(feat_preview.head(3).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
for label, ser in [('5日动量', mom5['TECH']), ('20日动量', mom20['TECH']), ('60日动量', mom60['TECH'])]:
    ax.plot(ser, lw=1, label=label)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_title('TECH 股票多周期动量特征（已 shift(1) 防前视）')
ax.set_ylabel('动量')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3：移动均线与价格偏离（MA/EMA）

ma5  = prices.rolling(5,  min_periods=5).mean()
ma20 = prices.rolling(20, min_periods=20).mean()
ma60 = prices.rolling(60, min_periods=60).mean()

pmr5  = (prices / ma5  - 1).shift(1)
pmr20 = (prices / ma20 - 1).shift(1)
pmr60 = (prices / ma60 - 1).shift(1)

ema12 = prices.ewm(span=12, adjust=False).mean()
ema26 = prices.ewm(span=26, adjust=False).mean()

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

axes[0].plot(prices['TECH'], lw=1.2, color='black', label='收盘价')
axes[0].plot(ma5['TECH'],   lw=1, color='#1f77b4', linestyle='--', label='MA5')
axes[0].plot(ma20['TECH'],  lw=1, color='#ff7f0e', linestyle='--', label='MA20')
axes[0].plot(ma60['TECH'],  lw=1, color='#2ca02c', linestyle='--', label='MA60')
axes[0].set_title('TECH 收盘价与移动均线')
axes[0].set_ylabel('价格')
axes[0].legend(fontsize=9)

axes[1].plot(pmr20['TECH'], lw=1, color='#ff7f0e', label='PMR20 (已shift)')
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].fill_between(pmr20.index, pmr20['TECH'], 0,
                     where=pmr20['TECH'] > 0, alpha=0.3, color='#2ca02c', label='价格>MA20')
axes[1].fill_between(pmr20.index, pmr20['TECH'], 0,
                     where=pmr20['TECH'] < 0, alpha=0.3, color='#d62728', label='价格<MA20')
axes[1].set_title('价格偏离 MA20 比率（PMR20，已 shift(1) 防前视）')
axes[1].set_ylabel('偏离率')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4：RSI 指标实现与可视化

def compute_rsi(px, window=14):
    delta = px.diff()
    gain  = delta.clip(lower=0).rolling(window, min_periods=window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window, min_periods=window).mean()
    rs    = gain / loss.replace(0.0, np.nan)
    rsi   = 100 - 100 / (1 + rs)
    rsi[loss == 0] = 100.0
    return rsi

rsi14 = compute_rsi(prices, 14)
rsi14_shifted = rsi14.shift(1)

print('RSI14 统计（shift 后）：')
print(rsi14_shifted.describe().round(2))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax1.plot(prices['TECH'], lw=1.2, color='black')
ax1.set_title('TECH 收盘价')
ax1.set_ylabel('价格')

ax2.plot(rsi14_shifted['TECH'], lw=1, color='#9467bd', label='RSI(14)（已shift）')
ax2.axhline(70, color='#d62728', lw=1.2, linestyle='--', label='超买线 70')
ax2.axhline(30, color='#2ca02c', lw=1.2, linestyle='--', label='超卖线 30')
ax2.axhline(50, color='gray', lw=0.8, linestyle=':')
ax2.fill_between(rsi14_shifted.index, rsi14_shifted['TECH'], 70,
                 where=rsi14_shifted['TECH'] > 70, alpha=0.25, color='#d62728')
ax2.fill_between(rsi14_shifted.index, rsi14_shifted['TECH'], 30,
                 where=rsi14_shifted['TECH'] < 30, alpha=0.25, color='#2ca02c')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI')
ax2.set_title('TECH RSI(14)（已 shift(1) 防前视）')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5：MACD 与布林带

def compute_macd(px, fast=12, slow=26, signal=9):
    ema_f = px.ewm(span=fast, adjust=False).mean()
    ema_s = px.ewm(span=slow, adjust=False).mean()
    dif   = ema_f - ema_s
    dea   = dif.ewm(span=signal, adjust=False).mean()
    return dif, dea, 2 * (dif - dea)

def compute_boll(px, window=20, k=2):
    mid   = px.rolling(window, min_periods=window).mean()
    sigma = px.rolling(window, min_periods=window).std()
    upper = mid + k * sigma
    lower = mid - k * sigma
    bw    = (upper - lower).replace(0, np.nan)
    return upper, mid, lower, (px - lower) / bw

dif, dea, macd_bar = compute_macd(prices)
dif_s  = dif.shift(1)
macd_s = macd_bar.shift(1)

boll_up, boll_mid, boll_lo, pct_b = compute_boll(prices)
pct_b_s = pct_b.shift(1)

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

axes[0].plot(prices['TECH'], lw=1.2, color='black', label='收盘价')
axes[0].plot(boll_mid['TECH'], lw=1, color='#ff7f0e', linestyle='--', label='中轨 MA20')
axes[0].fill_between(prices.index, boll_up['TECH'], boll_lo['TECH'],
                     alpha=0.15, color='steelblue', label='布林带')
axes[0].set_title('TECH 布林带（展示形态）')
axes[0].set_ylabel('价格')
axes[0].legend(fontsize=9)

axes[1].plot(pct_b_s['TECH'], lw=1, color='#9467bd', label='%B（已shift）')
axes[1].axhline(1, color='#d62728', lw=1, linestyle='--', label='上轨 %B=1')
axes[1].axhline(0, color='#2ca02c', lw=1, linestyle='--', label='下轨 %B=0')
axes[1].axhline(0.5, color='gray', lw=0.8, linestyle=':')
axes[1].set_title('布林带 %B 指标（已 shift(1) 防前视）')
axes[1].set_ylabel('%B')
axes[1].legend(fontsize=9)

colors_macd = ['#d62728' if v >= 0 else '#2ca02c' for v in macd_s['TECH']]
axes[2].bar(macd_s.index, macd_s['TECH'], color=colors_macd, alpha=0.6, width=1)
axes[2].plot(dif_s['TECH'], lw=1, color='#1f77b4', label='DIF（已shift）')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_title('TECH MACD（已 shift(1) 防前视）')
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('各技术指标已全部应用 shift(1)，无前视偏差。')

## 前视偏差对照实验

- **含前视**：直接用 `pct_change(20)`，包含 $t$ 日信息
- **无前视**：`pct_change(20).shift(1)`，只含 $t-1$ 日及更早信息

通过比较两种特征与次日收益的截面 IC，量化前视偏差的危害。

In [ ]:
# Cell 6：前视偏差对照实验

ret_next = rets.shift(-1)

mom20_biased = prices.pct_change(20)           # 含 t 日信息（错误）
mom20_clean  = prices.pct_change(20).shift(1)  # 只含 t-1 日及更早（正确）

def rolling_ic(feat, target):
    f, t = feat.align(target, join='inner')
    return f.corrwith(t, axis=1, method='pearson')

ic_biased = rolling_ic(mom20_biased, ret_next).dropna()
ic_clean  = rolling_ic(mom20_clean,  ret_next).dropna()

ic_ir_b = ic_biased.mean() / ic_biased.std() if ic_biased.std() != 0 else float('nan')
ic_ir_c = ic_clean.mean()  / ic_clean.std()  if ic_clean.std()  != 0 else float('nan')

print('=' * 55)
print(f'               含前视版本     无前视版本')
print(f'平均 IC        {ic_biased.mean():.4f}         {ic_clean.mean():.4f}')
print(f'IC 标准差      {ic_biased.std():.4f}         {ic_clean.std():.4f}')
print(f'ICIR           {ic_ir_b:.4f}         {ic_ir_c:.4f}')
print(f'|IC|>0.05 占比  {(ic_biased.abs()>0.05).mean():.1%}         {(ic_clean.abs()>0.05).mean():.1%}')
print('=' * 55)
print('>> 含前视版本的 IC 明显虚高，无前视版本才是真实预测能力。')

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(ic_biased, lw=0.8, color='#d62728', alpha=0.7,
             label=f'含前视 IC（均值={ic_biased.mean():.3f}）')
axes[0].axhline(ic_biased.mean(), color='#d62728', lw=1.5, linestyle='--')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('【含前视偏差】20 日动量 IC（错误做法）')
axes[0].set_ylabel('IC')
axes[0].legend(fontsize=10)

axes[1].plot(ic_clean, lw=0.8, color='#2ca02c', alpha=0.7,
             label=f'无前视 IC（均值={ic_clean.mean():.3f}）')
axes[1].axhline(ic_clean.mean(), color='#2ca02c', lw=1.5, linestyle='--')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('【无前视偏差】20 日动量 IC（正确做法）')
axes[1].set_ylabel('IC')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7：截面 Winsorize 与 z-score 标准化

def cross_winsorize(df, lower=0.05, upper=0.95):
    q_lo = df.quantile(lower, axis=1)
    q_hi = df.quantile(upper, axis=1)
    return df.clip(lower=q_lo, upper=q_hi, axis=0)

def cross_zscore(df):
    mean_ = df.mean(axis=1)
    std_  = df.std(axis=1).replace(0, np.nan)
    return df.sub(mean_, axis=0).div(std_, axis=0)

def cross_rank(df):
    return df.rank(axis=1, pct=True)

mom20_raw    = mom20.dropna()
mom20_wins   = cross_winsorize(mom20_raw)
mom20_zscore = cross_zscore(mom20_wins)

print('=== mom20 原始统计 ===')
print(mom20_raw.describe().round(4).to_string())
print()
print('=== mom20 截面 z-score 后统计 ===')
print(mom20_zscore.describe().round(4).to_string())

ts_mean = mom20_raw['TECH'].mean()
ts_std  = mom20_raw['TECH'].std()
mom20_ts_z = (mom20_raw['TECH'] - ts_mean) / ts_std

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(mom20_raw['TECH'], lw=1, color='black', label='原始 mom20')
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].set_title('原始 20 日动量特征（已 shift）')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend(fontsize=9)

axes[1].plot(mom20_ts_z, lw=1, color='#1f77b4', label='时序 z-score')
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_title('时序 z-score（全样本均值/标准差）')
axes[1].legend(fontsize=9)

axes[2].plot(mom20_zscore['TECH'], lw=1, color='#ff7f0e', label='截面 z-score')
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_title('截面 z-score（每日 4 只股票截面标准化，取 TECH）')
axes[2].legend(fontsize=9)

fig.suptitle('三种标准化方式对比', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('截面 z-score 消除了时间趋势，截面信号更稳定可比。')

In [ ]:
# Cell 8：构建完整特征矩阵并计算 IC/ICIR

features_dict = {
    'mom5':   mom5,
    'mom20':  mom20,
    'mom60':  mom60,
    'rev5':   rev5,
    'vol20':  vol20,
    'rsi14':  rsi14_shifted,
    'pmr20':  pmr20,
    'pct_b':  pct_b_s,
    'dif':    dif_s,
    'macd':   macd_s,
}

feat_for_ic = {name: cross_zscore(cross_winsorize(df)) for name, df in features_dict.items()}
ret_next_clean = rets.shift(-1)

ic_table = {}
for fname, feat in feat_for_ic.items():
    ic_ts = feat.corrwith(ret_next_clean, axis=1, method='pearson').dropna()
    ic_table[fname] = {
        'IC均值':   round(float(ic_ts.mean()), 4),
        'IC标准差': round(float(ic_ts.std()),  4),
        'ICIR':     round(float(ic_ts.mean() / ic_ts.std()) if ic_ts.std() != 0 else float('nan'), 4),
        'IC>0占比': round(float((ic_ts > 0).mean()), 4),
    }

df_ic = pd.DataFrame(ic_table).T.sort_values('ICIR', ascending=False)
print('=== 各特征 IC / ICIR 汇总（截面 Pearson，次日收益）===')
print(df_ic.to_string())
print()
valid = df_ic[df_ic['IC均值'].abs() > 0.02]
print(f'通过 |IC| > 0.02 筛选：{list(valid.index)}')

In [ ]:
# Cell 9：特征相关热力图

feat_ts = pd.DataFrame(
    {name: cross_zscore(cross_winsorize(df)).mean(axis=1)
     for name, df in features_dict.items()}
).dropna()

corr_feat = feat_ts.corr()
n = len(corr_feat)
feat_names = list(corr_feat.columns)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_feat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson 相关系数', shrink=0.8)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(feat_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(feat_names, fontsize=9)
ax.set_title('特征相关性热力图（截面均值时序）', fontsize=12)
for i in range(n):
    for j in range(n):
        val = corr_feat.iloc[i, j]
        c = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=c)
plt.tight_layout()
plt.show()

print('=== 高度相关特征对（|r| > 0.7）===')
upper_tri = corr_feat.where(np.triu(np.ones_like(corr_feat, dtype=bool), k=1))
high_corr = [
    (r, c, round(float(upper_tri.loc[r, c]), 3))
    for r in upper_tri.index
    for c in upper_tri.columns
    if pd.notna(upper_tri.loc[r, c]) and abs(upper_tri.loc[r, c]) > 0.7
]
if high_corr:
    for r, c, v in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {r:8s} x {c:8s}: r = {v:.3f}')
else:
    print('  无高度相关特征对（|r| <= 0.7）')

In [ ]:
# Cell 10：IC 衰减曲线

max_lag = 10
selected_feats = ['mom20', 'vol20', 'rsi14', 'mom5']

ic_decay = {}
for fname in selected_feats:
    feat = feat_for_ic[fname]
    lags = []
    for lag in range(1, max_lag + 1):
        ret_lag = rets.shift(-lag)
        ic_lag  = float(feat.corrwith(ret_lag, axis=1, method='pearson').dropna().mean())
        lags.append(ic_lag)
    ic_decay[fname] = lags

df_decay = pd.DataFrame(ic_decay, index=range(1, max_lag + 1))

fig, ax = plt.subplots(figsize=(10, 5))
for col in df_decay.columns:
    ax.plot(df_decay.index, df_decay[col], marker='o', markersize=5, lw=1.5, label=col)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.axhline(0.02,  color='gray', lw=0.8, linestyle=':', alpha=0.7)
ax.axhline(-0.02, color='gray', lw=0.8, linestyle=':', alpha=0.7)
ax.set_xlabel('预测目标的滞后期（天）')
ax.set_ylabel('平均 IC')
ax.set_title('IC 衰减曲线：特征预测力随预测期的变化')
ax.legend(fontsize=10)
ax.set_xticks(range(1, max_lag + 1))
plt.tight_layout()
plt.show()
print('IC 衰减越快，特征有效预测期越短；稳健特征在滞后 1~3 期内仍有 IC 强度。')

In [ ]:
# Cell 11：多重共线性 VIF 诊断

import statsmodels.api as sm

feat_ts_clean = feat_ts[selected_feats].dropna()

def compute_vif(X_df):
    rows = []
    for col in X_df.columns:
        others = sm.add_constant(X_df.drop(columns=[col]))
        r2  = sm.OLS(X_df[col], others).fit().rsquared
        vif = 1 / (1 - r2) if r2 < 1 else float('inf')
        rows.append({'特征': col, 'R2': round(r2, 4), 'VIF': round(vif, 3)})
    return pd.DataFrame(rows).set_index('特征')

df_vif = compute_vif(feat_ts_clean)
print('=== VIF 诊断结果 ===')
print(df_vif.to_string())
print()
print('VIF < 5 正常；5~10 中度；> 10 严重共线性')
high_vif = df_vif[df_vif['VIF'] > 5]
if len(high_vif) > 0:
    print(f'中度/严重共线性特征：{list(high_vif.index)}')
else:
    print('所选特征无严重多重共线性（VIF <= 5）')

In [ ]:
# Cell 12：方差阈值与相关性特征筛选
# 注意：方差阈值筛选在原始（未标准化）特征上进行，标准化后特征方差接近 1，不适合用此方法

from sklearn.feature_selection import VarianceThreshold

# 构建原始特征矩阵（未标准化，用于方差阈值演示）
raw_feat_ts = pd.DataFrame(
    {name: df.mean(axis=1) for name, df in features_dict.items()}
).dropna()

print(f'筛选前特征数：{raw_feat_ts.shape[1]}')
print('原始特征方差（跨时间序列）：')
print(raw_feat_ts.var().round(6).to_string())
print()

# 动态选择合理阈值（取最小方差的一半，保证至少保留 1 个特征）
min_var = float(raw_feat_ts.var().min())
threshold = min_var * 0.1
sel = VarianceThreshold(threshold=threshold)
sel.fit(raw_feat_ts)
kept_cols = raw_feat_ts.columns[sel.get_support()].tolist()
print(f'方差阈值（{threshold:.2e}）筛选后保留：{kept_cols}')
print()

ic_means = df_ic['IC均值'].abs()

def remove_high_corr_feats(feat_df, ic_abs, threshold=0.8):
    corr = feat_df.corr().abs()
    upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
    to_drop = set()
    for col in upper.columns:
        high = upper[col][upper[col] > threshold]
        for other in high.index:
            drop_one = col if ic_abs.get(col, 0) < ic_abs.get(other, 0) else other
            to_drop.add(drop_one)
    return [c for c in feat_df.columns if c not in to_drop]

# 基于 feat_ts（标准化后，用于 IC 和相关性分析）
feat_matrix = feat_ts.dropna()
common_feats = [f for f in feat_matrix.columns if f in ic_means.index]
if common_feats:
    kept_after_corr = remove_high_corr_feats(
        feat_matrix[common_feats], ic_means[common_feats], threshold=0.8
    )
    print(f'相关性筛除（|r|>0.8）后保留：{kept_after_corr}')

## 本章小结

| 核心要点 | 关键写法 |
|---|---|
| 防前视偏差 | `feat = xxx.rolling(n).func().shift(1)` |
| 截面标准化 | `df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)` |
| IC 计算 | `feat.corrwith(ret_next, axis=1).mean()` |
| 高相关筛除 | `corr().abs() > 0.8` 删 IC 低者 |
| VIF 诊断 | `1 / (1 - r2)`，> 10 则处理 |

**金融特征工程核心法则**：时间对齐第一，信号质量第二，特征数量第三。

---
## 习题参考解答

以下对应正文第 10 章习题，可直接运行。

In [ ]:
# === 习题 10.1：前视偏差诊断 ===
print('习题 10.1：前视偏差诊断')
print('=' * 50)

feat_a = prices.rolling(20).mean()
feat_b = prices.rolling(20).mean().shift(1)
feat_c = prices.pct_change(20).shift(1)

ret_q1 = rets.shift(-1)
ic_a = feat_a.corrwith(ret_q1, axis=1).dropna().mean()
ic_b = feat_b.corrwith(ret_q1, axis=1).dropna().mean()
ic_c = feat_c.corrwith(ret_q1, axis=1).dropna().mean()

print(f'feat_a（含当日均线）IC：{ic_a:.4f}')
print(f'feat_b（shift(1) 均线）IC：{ic_b:.4f}')
print(f'feat_c（shift(1) 动量）IC：{ic_c:.4f}')
print()
print('(a) feat_a 在开盘前选股时有前视偏差；feat_b/feat_c 均安全。')
print('(b) feat_a 在收盘后使用（t 日数据已知）则无前视偏差。')
print('(c) 开盘前选股：feat_b 和 feat_c 均安全，建议使用 feat_c（动量直观）。')

In [ ]:
# === 习题 10.2：RSI 实现与双轴图 ===
print('习题 10.2：RSI(14) 实现与双轴图')

rsi_tech   = rsi14_shifted['TECH'].dropna()
price_tech = prices['TECH'].loc[rsi_tech.index]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(price_tech, lw=1.2, color='black')
ax1.set_title('TECH 收盘价')
ax1.set_ylabel('价格')

ax2.plot(rsi_tech, lw=1, color='#9467bd')
ax2.axhline(70, color='#d62728', lw=1.5, linestyle='--', label='超买 70')
ax2.axhline(30, color='#2ca02c', lw=1.5, linestyle='--', label='超卖 30')
ax2.axhline(50, color='gray', lw=0.8, linestyle=':')
ax2.fill_between(rsi_tech.index, rsi_tech, 70, where=rsi_tech > 70,
                 alpha=0.3, color='#d62728', label='超买区间')
ax2.fill_between(rsi_tech.index, rsi_tech, 30, where=rsi_tech < 30,
                 alpha=0.3, color='#2ca02c', label='超卖区间')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI(14)（已shift）')
ax2.set_title('TECH RSI(14) —— 习题 10.2')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'RSI: max={rsi_tech.max():.1f}, min={rsi_tech.min():.1f}, mean={rsi_tech.mean():.1f}')

In [ ]:
# === 习题 10.3：截面 vs 时序标准化 ===
print('习题 10.3：截面 vs 时序标准化')

ts_z = (mom20['TECH'] - mom20['TECH'].mean()) / mom20['TECH'].std()
cs_z = cross_zscore(mom20)['TECH']

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
common_idx = mom20['TECH'].dropna().index

axes[0].plot(mom20.loc[common_idx, 'TECH'], lw=1, color='black')
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].set_title('原始 20 日动量（TECH）')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

axes[1].plot(ts_z.loc[common_idx], lw=1, color='#1f77b4', label='时序 z-score')
axes[1].axhline(0, color='gray', lw=0.8)
axes[1].set_title('时序 z-score（全样本均值/标准差）')
axes[1].legend()

axes[2].plot(cs_z.loc[common_idx], lw=1, color='#ff7f0e', label='截面 z-score')
axes[2].axhline(0, color='gray', lw=0.8)
axes[2].set_title('截面 z-score（每日 4 只股票截面标准化，取 TECH）')
axes[2].legend()

fig.suptitle('习题 10.3：时序 vs 截面标准化对比', fontsize=12)
plt.tight_layout()
plt.show()

print()
print('(c) 截面 z-score 适合截面选股：')
print('    策略目标是某天选哪些股票，关注截面相对排名而非时序水平。')
print('    牛市整体动量高，时序标准化会使截面信号失真。')
print('    截面 z-score 每天强制均值=0、标准差=1，不同时期截面信号可比。')

In [ ]:
# === 习题 10.4：IC 计算与特征筛选 ===
print('习题 10.4：4 个特征 IC/ICIR')

ex4_feats = {
    '5日动量':   mom5,
    '20日动量':  mom20,
    '20日波动率': vol20,
    'RSI14':    rsi14_shifted,
}

ret_q4 = rets.shift(-1)
ic_ex4 = {}
for fname, feat in ex4_feats.items():
    ic_ts = feat.corrwith(ret_q4, axis=1, method='spearman').dropna()
    ic_ex4[fname] = {
        '平均IC':  round(float(ic_ts.mean()), 4),
        'IC标准差': round(float(ic_ts.std()),  4),
        'ICIR':   round(float(ic_ts.mean() / ic_ts.std()) if ic_ts.std() != 0 else float('nan'), 4),
        '|IC|>0.02': round(float((ic_ts.abs() > 0.02).mean()), 3),
    }

df_ex4 = pd.DataFrame(ic_ex4).T
print('(a) Spearman IC/ICIR：')
print(df_ex4.to_string())
print()

passed = [f for f, row in ic_ex4.items() if abs(row['平均IC']) > 0.02]
print(f'(b) |平均IC| > 0.02：{passed if passed else "无（样本量小，IC 有限）"}')
print()

feat_for_corr = pd.concat(
    {fname: feat.mean(axis=1) for fname, feat in ex4_feats.items()}, axis=1
).dropna()
print('(c) 特征相关矩阵：')
print(feat_for_corr.corr().round(3).to_string())

In [ ]:
# === 习题 10.5：布林带 %B 特征分析 ===
print('习题 10.5：布林带 %B')

pct_b_tech = pct_b_s['TECH'].dropna()
ret_q5_tech = rets['TECH'].shift(-1).loc[pct_b_tech.index]
valid_idx = pct_b_tech.dropna().index.intersection(ret_q5_tech.dropna().index)
ic_pctb = float(np.corrcoef(
    pct_b_tech.loc[valid_idx].values,
    ret_q5_tech.loc[valid_idx].values
)[0, 1])

print(f'(a) %B: mean={pct_b_tech.mean():.3f}, std={pct_b_tech.std():.3f}')
print(f'(b) %B 与次日收益 IC（TECH）：{ic_pctb:.4f}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(prices['TECH'].loc[pct_b_tech.index], lw=1.2, color='black', label='收盘价')
ax1.plot(boll_mid.loc[pct_b_tech.index, 'TECH'], lw=1, linestyle='--',
         color='#ff7f0e', label='MA20 中轨')
ax1.fill_between(pct_b_tech.index,
                 boll_up.loc[pct_b_tech.index, 'TECH'],
                 boll_lo.loc[pct_b_tech.index, 'TECH'],
                 alpha=0.15, color='steelblue', label='布林带')
ax1.set_title('TECH 布林带')
ax1.set_ylabel('价格')
ax1.legend(fontsize=9)

ax2.plot(pct_b_tech, lw=1, color='#9467bd', label=f'%B（已shift，IC={ic_pctb:.4f}）')
ax2.axhline(1.0, color='#d62728', lw=1.2, linestyle='--', label='上轨 %B=1')
ax2.axhline(0.0, color='#2ca02c', lw=1.2, linestyle='--', label='下轨 %B=0')
ax2.axhline(0.5, color='gray', lw=0.8, linestyle=':')
ax2.set_title('TECH %B 指标（已shift）')
ax2.set_ylabel('%B')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print()
print('(c) %B 双重解读：')
print('    趋势市：%B > 1（突破上轨）-> 续涨（动量）')
print('    震荡市：%B > 1（超买）-> 回调（反转）')
print('    判断依据：结合长周期趋势过滤器（如 MA60 方向）。')